# 01 · UniProt Data Inspection

Goal: understand the reviewed human protein dataset before cleaning it.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent

uniprot_path = ROOT / "data/raw/uniprot/uniprotkb_organism_id_9606_AND_reviewed_2026_10_12.tsv.gz"

print("Dataset path:", uniprot_path)
print("File exists :", uniprot_path.exists())
print(f"File size   : {uniprot_path.stat().st_size / 1024**2:.2f} MB")

Dataset path: c:\Users\rajdeep\Desktop\Capstone Project_GNN\data\raw\uniprot\uniprotkb_organism_id_9606_AND_reviewed_2026_10_12.tsv.gz
File exists : True
File size   : 18.45 MB


In [2]:
uniprot_df = pd.read_csv(
    uniprot_path,
    sep="\t",
    compression="gzip",
    dtype=str
)

print("Shape:", uniprot_df.shape)
print("\nColumns:")
for column in uniprot_df.columns:
    print("-", column)

Shape: (20431, 18)

Columns:
- Entry
- Entry Name
- Gene Names (primary)
- Protein names
- Length
- Sequence
- EC number
- Keywords
- Gene Ontology IDs
- Gene Ontology (biological process)
- Gene Ontology (molecular function)
- Gene Ontology (cellular component)
- Function [CC]
- Subcellular location [CC]
- Involvement in disease
- MIM
- STRING
- Reactome


In [3]:
important_columns = [
    "Entry",
    "Entry Name",
    "Gene Names (primary)",
    "Protein names",
    "Length",
    "Sequence",
    "Gene Ontology IDs",
    "Involvement in disease",
    "STRING",
]

uniprot_df[important_columns].head(5)

,Entry,Entry Name,Gene Names (primary),Protein names,Length,Sequence,Gene Ontology IDs,Involvement in disease,STRING
0,A0A087X1C5,CP2D7_HUMAN,CYP2D7,Cytochrome P450 2D7 (EC 1.14.14.1),515,MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,GO:0005506; GO:0005737; GO:0005739; GO:0006805...,NaN,NaN
1,A0A096LP01,SIM26_HUMAN,SMIM26,Small integral membrane protein 26,95,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,GO:0005739; GO:0005741; GO:0019901; GO:0044325,NaN,9606.ENSP00000485316;
2,A0A0B4J2F0,PIOS1_HUMAN,PIGBOS1,Protein PIGBOS1 (PIGB opposite strand protein 1),54,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,GO:0005739; GO:0005741; GO:1900101,NaN,9606.ENSP00000484893;
3,A0A0C5B5G6,MOTSC_HUMAN,MT-RNR1,Mitochondrial-derived peptide MOTS-c (Mitochon...,16,MRWQEMGYIFYPRKLR,GO:0001649; GO:0003677; GO:0005576; GO:0005634...,NaN,NaN
4,A0A0K2S4Q6,CD3CH_HUMAN,CD300H,Protein CD300H (CD300 antigen-like family memb...,201,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...,GO:0004888; GO:0005576; GO:0005886; GO:0007165...,NaN,NaN


In [4]:
print("Total proteins:", len(uniprot_df))
print("Unique UniProt IDs:", uniprot_df["Entry"].nunique())

print("\nMissing values:")
for column in important_columns:
    missing = uniprot_df[column].isna().sum()
    print(f"{column:30s}: {missing:,}")

sequence_lengths = uniprot_df["Sequence"].str.len()

print("\nSequence length statistics:")
print(sequence_lengths.describe())

has_go = uniprot_df["Gene Ontology IDs"].fillna("").str.strip().ne("").sum()
has_string = uniprot_df["STRING"].fillna("").str.strip().ne("").sum()
has_disease = uniprot_df["Involvement in disease"].fillna("").str.strip().ne("").sum()

print("\nAnnotation coverage:")
print(f"Proteins with GO labels       : {has_go:,}")
print(f"Proteins with STRING IDs      : {has_string:,}")
print(f"Proteins with disease text    : {has_disease:,}")

Total proteins: 20431
Unique UniProt IDs: 20431

Missing values:
Entry                         : 0
Entry Name                    : 0
Gene Names (primary)          : 148
Protein names                 : 0
Length                        : 0
Sequence                      : 0
Gene Ontology IDs             : 1,056
Involvement in disease        : 15,090
STRING                        : 1,590

Sequence length statistics:
count    20431.000000
mean       558.868239
std        608.436061
min          2.000000
25%        249.000000
50%        415.000000
75%        670.000000
max      34350.000000
Name: Sequence, dtype: float64

Annotation coverage:
Proteins with GO labels       : 19,375
Proteins with STRING IDs      : 18,841
Proteins with disease text    : 5,341


In [5]:
example = uniprot_df.loc[
    uniprot_df["Gene Names (primary)"].eq("TP53"),
    important_columns
]

example

,Entry,Entry Name,Gene Names (primary),Protein names,Length,Sequence,Gene Ontology IDs,Involvement in disease,STRING
2078,P04637,P53_HUMAN,TP53,Cellular tumor antigen p53 (Antigen NY-CO-13) ...,393,MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLS...,GO:0000122; GO:0000785; GO:0000976; GO:0000978...,DISEASE: Note=TP53 is found in increased amoun...,9606.ENSP00000269305;


In [6]:
print("Total proteins:", len(uniprot_df))
print("Unique UniProt IDs:", uniprot_df["Entry"].nunique())

print("\nMissing values:")
for column in important_columns:
    missing = uniprot_df[column].isna().sum()
    print(f"{column:30s}: {missing:,}")

sequence_lengths = uniprot_df["Sequence"].str.len()

print("\nSequence length statistics:")
print(sequence_lengths.describe())

has_go = uniprot_df["Gene Ontology IDs"].fillna("").str.strip().ne("").sum()
has_string = uniprot_df["STRING"].fillna("").str.strip().ne("").sum()
has_disease = uniprot_df["Involvement in disease"].fillna("").str.strip().ne("").sum()

print("\nAnnotation coverage:")
print(f"Proteins with GO labels       : {has_go:,}")
print(f"Proteins with STRING IDs      : {has_string:,}")
print(f"Proteins with disease text    : {has_disease:,}")

Total proteins: 20431
Unique UniProt IDs: 20431

Missing values:
Entry                         : 0
Entry Name                    : 0
Gene Names (primary)          : 148
Protein names                 : 0
Length                        : 0
Sequence                      : 0
Gene Ontology IDs             : 1,056
Involvement in disease        : 15,090
STRING                        : 1,590

Sequence length statistics:
count    20431.000000
mean       558.868239
std        608.436061
min          2.000000
25%        249.000000
50%        415.000000
75%        670.000000
max      34350.000000
Name: Sequence, dtype: float64

Annotation coverage:
Proteins with GO labels       : 19,375
Proteins with STRING IDs      : 18,841
Proteins with disease text    : 5,341
